# Web Information Retrieval WS 2026 - PyTerrier Tutorial - Part 3

In this third tutorial, we want to generate initial rankings and then evaluate them.

For this, we continue using PyTerrier and the LongEval-Sci test collection, which was indexed in the first tutorial and analyzed in the second tutorial.

In [ ]:
import pandas as pd
import pyterrier as pt

We load the index again.

In [ ]:
index = pt.IndexFactory.of("./index")
print(index.getCollectionStatistics().toString())

Next, we define the two search systems Tf and BM25.

In [ ]:
tf = pt.BatchRetrieve(index, wmodel="Tf")
bm25 = pt.BatchRetrieve(index, wmodel="BM25")

We can now use these systems to search the test collection. For this, we use the `search` function. As a result, we obtain a dataframe containing the ranked documents.

In [ ]:
tf.search("Hello World")

In [ ]:
# TODO:
# Search the systeme with different queries.

To compare the two systems, we need to conduct an experiment. Conveniently, PyTerrier helps with this. The `experiment` function does exactly that. As input, we provide:

- `retr_systems`: A list of retrieval systems that we want to compare.
- `topics`: The queries used to perform the search.
- `qrel`: The relevance assessments from the test collection.
- `eval_metrics`: A list of retrieval metrics used to measure effectiveness.

An overview of the supported metrics can be found in the [documentation](https://pyterrier.readthedocs.io/en/latest/experiments.html#available-evaluation-measures).

In [ ]:
from ir_datasets_longeval import load

dataset = load("longeval-sci-2026/snapshot-1/train/dctr")

In [ ]:
topics = pd.DataFrame(dataset.queries)
topics.rename(columns={"query_id": "qid", "text": "query"}, inplace=True)

topics = pd.DataFrame(
    [{"qid": i.query_id, "query": i.default_text()} for i in dataset.queries_iter()]
)

# PyTerrier needs to use pre-tokenized queries
tokeniser = pt.java.autoclass(
    "org.terrier.indexing.tokenisation.Tokeniser"
).getTokeniser()

topics["query"] = topics["query"].apply(lambda i: " ".join(tokeniser.getTokens(i)))


In [ ]:
qrels = pd.DataFrame(dataset.qrels)
qrels.rename(columns={"query_id": "qid", "doc_id": "docno"}, inplace=True)

In [ ]:
results = pt.Experiment(
    retr_systems=[tf, bm25],
    topics=topics,
    qrels=qrels,
    eval_metrics=["ndcg"],
    verbose = True
)
results

In [ ]:
# TODO:
# Do some experiments and compare the outcomes using different measures
# Can you obsorve different behaviour ddue to the different measures?

To gain a more detailed insight into the results, we now output the results per topic, i.e., per search query. These results allow for further analyses. Interesting questions include:
- For which topic is a system most effective and for which is it least effective?
- What could be the reason for this?
- Are there areas in which one system performs particularly well?
- How can the results be visualized?

In [ ]:
results = pt.Experiment(
    retr_systems=[tf, bm25],
    topics=topics,
    qrels=qrels,
    perquery=True,
    eval_metrics=["ndcg_cut_10"]
)
results

In [ ]:
# TODO
# Play around, use your own topics!

In [ ]:
topics